# B2 — Análise Semântica de Drift (global) — Colab

Adaptador fino para o escopo **`global`**. Monta o Google Drive, atualiza o repositório, instala dependências e chama `scripts/drift/run_b2.py` em loop sobre as duas granularidades (mensal + bi-semanal). **Nenhuma lógica científica vive neste notebook** (CLAUDE.md §5).

Aplica três métricas semânticas sobre o cache de embeddings BERTimbau `[CLS]` (ADR 011 §D.1):
- **Cosseno consecutivo** entre centróides de janelas adjacentes;
- **Cosseno cumulativo** contra média histórica dos centróides;
- **MMD²** via `MMDDrift(n_permutations=1)`, retornando só a estatística (kernel RBF, bandwidth via mediana).

Filtro R2 (ADR 011) aplicado na origem em `src/drift/janelas.filtrar_periodo_efetivo` — `2017-10` fica fora simultaneamente em `time_ordered` e no baseline `randomized`.

**Pré-requisitos:**
- Embeddings já gerados (`compute_embeddings_drift.ipynb`) em `MyDrive/ptbr-market-classification/artifacts/drift/embeddings/bertimbau_base_cls/`.
- `data/processado/corpus_opcao7.parquet` no Drive (para `y_original`).
- Runtime: GPU recomendada (T4/L4) para acelerar MMD². CPU também funciona, com tempo maior.

**Saída** em `MyDrive/ptbr-market-classification/artifacts/drift/b2_semantic/`:
- Um diretório por combo (`<timestamp>-<granularidade>-global/`) com `metadata.json` + `results.parquet`.

**Tempo esperado** (estimativa grosseira, GPU L4):
- Mensal (global): ~5–10 min × 6 condições (1 time + 5 rand) = ~30–60 min.
- Bi-semanal (global): ~10–20 min × 6 condições = ~60–120 min.


## 1. Parâmetros (editar conforme necessário)


In [ ]:
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification-2.git'  # substituir pela URL do seu fork
RAMO = 'main'

DIR_REPO = '/content/ptbr-market-classification'
DIR_DRIVE = '/content/drive/MyDrive/ptbr-market-classification'

CAMINHO_EMBEDDINGS = f'{DIR_DRIVE}/artifacts/drift/embeddings/bertimbau_base_cls/embeddings.parquet'
CAMINHO_CORPUS = f'{DIR_DRIVE}/data/processado/corpus_opcao7.parquet'
DIR_ARTEFATOS_DRIFT = f'{DIR_DRIVE}/artifacts/drift'

# Escopo fixo deste notebook. Para rodar os outros escopos, abra os
# notebooks irmãos `run_b2_semantic_drift_*.ipynb`.
ESCOPO = 'global'

# Granularidades a rodar neste escopo. Cada uma vira um diretório
# próprio em `artifacts/drift/b2_semantic/`. Comente o que não quiser
# executar.
GRANULARIDADES = [
    'mensal',
    'bisemanal',
]


## 2. Montar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clonar / atualizar repositório


In [ ]:
import os, subprocess

if os.path.exists(DIR_REPO):
    subprocess.run(['git', '-C', DIR_REPO, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'checkout', RAMO], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', RAMO, REPO_URL, DIR_REPO], check=True)

os.chdir(DIR_REPO)
print('cwd =', os.getcwd())


## 4. Instalar dependências


In [ ]:
!pip install -q -r requirements.txt


## 5. Validar GPU + inputs

GPU é recomendada mas não obrigatória — MMD² em CPU também funciona, só mais lento.


In [ ]:
import torch
from pathlib import Path

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} '
          f'({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB VRAM)')
    DEVICE = 'cuda'
else:
    print('GPU não detectada — rodando em CPU. Para acelerar MMD², ative '
          'Runtime → Change runtime type → GPU.')
    DEVICE = 'cpu'

for caminho, nome in [(CAMINHO_EMBEDDINGS, 'embeddings'), (CAMINHO_CORPUS, 'corpus')]:
    p = Path(caminho)
    assert p.exists(), f'{nome} não encontrado em {p}'
    print(f'{nome}: {p} ({p.stat().st_size / 1024**2:.1f} MB)')


## 6. Loop sobre granularidades (escopo = `global`)

Cada granularidade gera um diretório próprio em `artifacts/drift/b2_semantic/`. Falhas em uma granularidade não interrompem a próxima.


In [ ]:
import os, subprocess, time

os.environ['PTBR_MC_DIR_DRIFT'] = DIR_ARTEFATOS_DRIFT
os.environ['PTBR_MC_EMBEDDINGS_DRIFT'] = CAMINHO_EMBEDDINGS
os.environ['PTBR_MC_CORPUS_DRIFT'] = CAMINHO_CORPUS

falhas = []
for granularidade in GRANULARIDADES:
    print(f'\n{"="*70}\n>>> {granularidade} / {ESCOPO}\n{"="*70}')
    args = [
        'python', 'scripts/drift/run_b2.py',
        '--embeddings', CAMINHO_EMBEDDINGS,
        '--corpus', CAMINHO_CORPUS,
        '--out', DIR_ARTEFATOS_DRIFT,
        '--granularidade', granularidade,
        '--escopo', ESCOPO,
        '--device', DEVICE,
    ]
    t0 = time.perf_counter()
    res = subprocess.run(args, capture_output=False)
    dt = (time.perf_counter() - t0) / 60
    if res.returncode != 0:
        falhas.append((granularidade, ESCOPO, res.returncode))
        print(f'  FALHA (exit={res.returncode}) após {dt:.1f} min')
    else:
        print(f'  OK em {dt:.1f} min')

print(f'\n{"="*70}')
if falhas:
    print('Combos com falha:')
    for g, e, rc in falhas:
        print(f'  {g} / {e} (exit {rc})')
else:
    print(f'Todas as granularidades de `{ESCOPO}` OK.')


## 7. Resumo dos artefatos gerados (escopo = `global`)


In [ ]:
import json
from pathlib import Path
import pandas as pd

raiz_b2 = Path(DIR_ARTEFATOS_DRIFT) / 'b2_semantic'
print(f'Conteúdo de {raiz_b2} filtrado por escopo={ESCOPO}:')
for d in sorted(raiz_b2.glob(f'*-{ESCOPO}')):
    if not d.is_dir():
        continue
    parquet = d / 'results.parquet'
    metadata = d / 'metadata.json'
    if not (parquet.exists() and metadata.exists()):
        print(f'  {d.name}: INCOMPLETO')
        continue
    m = json.loads(metadata.read_text())
    df = pd.read_parquet(parquet, engine='pyarrow')
    # Média da métrica por condição (cosseno consecutivo apenas, como sanity check).
    cos = df[df['metrica'] == 'cosine_centroid']
    pt = cos[cos['condicao'] == 'time_ordered']['valor'].mean()
    pr = cos[cos['condicao'] == 'randomized']['valor'].mean()
    print(
        f'  {d.name}: {len(df)} linhas | '
        f'cos médio time-ordered={pt:.4f} vs randomized={pr:.4f} | '
        f'{m["duracao_segundos"] / 60:.1f} min'
    )
